# Pet Breed Classifier: train everything on a free GPU

Works on **Google Colab** (Runtime → Change runtime type → **T4 GPU**) or **Kaggle** (Settings → Accelerator: GPU, Internet: On).

Run the cells from top to bottom. The full experiment set takes about 1 hour.

In [ ]:
!nvidia-smi

## 1. Get the code
Option A (recommended): clone your GitHub repository. Option B: upload the project zip and unzip it.

In [ ]:
REPO = "https://github.com/giorgichokuri/pet-breed-classifier.git"  # change if your repo name differs
!git clone -q $REPO project || echo 'clone failed: upload the zip instead'
%cd project

In [ ]:
# Option B: after uploading pet-breed-classifier.zip in the file panel
# !unzip -q /content/pet-breed-classifier.zip -d /content && %cd /content/pet-breed-classifier

## 2. Install and run the tests

In [ ]:
!pip install -q -r requirements.txt
!python -m pytest -q

## 3. Run all experiments
Downloads the Oxford-IIIT Pet dataset (~800 MB) on the first run, trains six configurations, picks the best by validation accuracy, then exports, quantizes and explains it.

In [ ]:
!bash scripts/run_all.sh

## 4. Look at the results

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('experiments/RESULTS.md').read()))
display(Image('experiments/curves.png'))

In [ ]:
import subprocess
best = subprocess.check_output(['python', 'scripts/pick_best.py']).decode().strip()
print('best run:', best)
for name in ['confusion_matrix.png', 'misclassified.png', 'gradcam.png']:
    display(Image(f'experiments/{best}/{name}'))

## 5. Try the API right here (optional)

In [ ]:
import subprocess, time, requests
server = subprocess.Popen(['uvicorn', 'app.main:app', '--port', '8000'])
time.sleep(5)
img = 'data/oxford-iiit-pet/images/Abyssinian_1.jpg'
print(requests.post('http://localhost:8000/predict', files={'file': open(img, 'rb')}).json())
server.terminate()

## 6. Download the outputs
Then copy `experiments/` and `models/` into your local repository and push them to GitHub.

In [ ]:
!zip -qr outputs.zip experiments models -x 'experiments/*/best.pt'
try:
    from google.colab import files
    files.download('outputs.zip')
except ImportError:
    print('On Kaggle: download outputs.zip from the Output panel')